# 第五题：基于 EM 算法的 Iris 分类器

对应实验 7.5。使用 `sklearn.mixture.GaussianMixture` 通过 EM 学习三个高斯混合分量，
并计算 Iris 分类的 accuracy、宏平均查准率、宏平均查全率和宏平均 F1。

## 算法及类别映射

对连续属性建立高斯混合分布：

$$p(x)=\sum_{k=1}^{K}\pi_k\,\mathcal N(x\mid\mu_k,\Sigma_k),\qquad K=3.$$

E 步计算每个样本属于各分量的责任度：

$$r_{ik}=\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}
{\sum_l\pi_l\mathcal N(x_i\mid\mu_l,\Sigma_l)}.$$

M 步据此更新参数，其中 $N_k=\sum_i r_{ik}$：

$$\pi_k=\frac{N_k}{N},\qquad
\mu_k=\frac{\sum_i r_{ik}x_i}{N_k},\qquad
\Sigma_k=\frac{\sum_i r_{ik}(x_i-\mu_k)(x_i-\mu_k)^T}{N_k}.$$

交替执行 E、M 步直至收敛。高斯混合分量的编号本身不代表真实类别，不能直接与类别编号比较。
本实验先在**训练特征**上无监督拟合 GMM，再仅用**训练标签**构造“分量 × 类别”计数表，
用匈牙利算法求一一映射，使训练集上的匹配数最大。验证标签仅用于计算指标。
每折独立拟合和映射，不用验证标签重新排列分量，避免评价泄漏。

使用 k-means++ 初始均值、完整协方差、十次初始化、固定随机种子，
以及对协方差对角线的微小正则项。
直接使用原始 Iris 特征，不在全数据上做标准化或拟合其他变换。


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

data_path = next(
    (base / "data" / "iris" / "iris.data"
     for base in (Path.cwd(), Path.cwd().parent)
     if (base / "data" / "iris" / "iris.data").is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("请从项目根目录或 solution 目录运行本 Notebook。")
# Iris 最后一列是字符串类别，不能把整张表直接读成浮点数。
iris = np.loadtxt(data_path, delimiter=",", dtype=str)
irisX = iris[:, :4].astype(float)
irisY = iris[:, 4]
feature_names = ["萼片长度", "萼片宽度", "花瓣长度", "花瓣宽度"]
assert irisX.shape == (150, 4)
assert len(np.unique(irisY)) == 3 and np.isfinite(irisX).all()
print(f"Iris: {irisX.shape[0]} 个样本，{irisX.shape[1]} 个特征")
print(dict(zip(*np.unique(irisY, return_counts=True))))


Iris: 150 个样本，4 个特征
{'Iris-setosa': 50, 'Iris-versicolor': 50, 'Iris-virginica': 50}


In [2]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.mixture import GaussianMixture
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from scipy.optimize import linear_sum_assignment

class EMGaussianMixtureClassifier(ClassifierMixin, BaseEstimator):
    """GMM 无监督拟合 + 仅依赖训练标签的分量到类别映射。"""

    def __init__(self, random_state=42, n_init=10, max_iter=300, reg_covar=1e-6):
        self.random_state = random_state
        self.n_init = n_init
        self.max_iter = max_iter
        self.reg_covar = reg_covar

    def fit(self, X, y):
        X, y = check_X_y(X, y, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.classes_, class_index = np.unique(y, return_inverse=True)
        self.gmm_ = GaussianMixture(
            n_components=len(self.classes_), covariance_type="full",
            init_params="k-means++",
            random_state=self.random_state, n_init=self.n_init,
            max_iter=self.max_iter, reg_covar=self.reg_covar,
        ).fit(X)
        train_components = self.gmm_.predict(X)
        self.contingency_ = np.zeros(
            (len(self.classes_), len(self.classes_)), dtype=int
        )
        np.add.at(self.contingency_, (train_components, class_index), 1)
        # linear_sum_assignment 默认最小化，故对匹配计数取负数。
        rows, columns = linear_sum_assignment(-self.contingency_)
        self.component_class_index_ = np.empty(len(self.classes_), dtype=int)
        self.component_class_index_[rows] = columns
        self.component_to_class_ = self.classes_[self.component_class_index_]
        return self

    def _check_features(self, X):
        check_is_fitted(self, "component_to_class_")
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("训练和预测的特征数不一致。")
        return X

    def predict(self, X):
        X = self._check_features(X)
        components = self.gmm_.predict(X)
        return self.component_to_class_[components]

    def predict_proba(self, X):
        X = self._check_features(X)
        responsibilities = self.gmm_.predict_proba(X)
        probabilities = np.empty_like(responsibilities)
        # 将分量概率列重新排列为 classes_ 的类别顺序。
        probabilities[:, self.component_class_index_] = responsibilities
        return probabilities


## 分层十折评价

指标以各折的算术平均值及样本标准差报告。三分类的 precision、recall 和 F1 使用 macro 平均。
每次在九折训练，剩下一折只进行预测和评价；每个样本恰好得到一次折外预测。


In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
records = []
oof_prediction = np.empty_like(irisY)
trained_models = []
for fold, (train_index, valid_index) in enumerate(cv.split(irisX, irisY), 1):
    model = EMGaussianMixtureClassifier().fit(irisX[train_index], irisY[train_index])
    truth = irisY[valid_index]
    predicted = model.predict(irisX[valid_index])
    oof_prediction[valid_index] = predicted
    records.append({
        "fold": fold,
        "accuracy": accuracy_score(truth, predicted),
        "precision_macro": precision_score(truth, predicted, average="macro", zero_division=0),
        "recall_macro": recall_score(truth, predicted, average="macro", zero_division=0),
        "f1_macro": f1_score(truth, predicted, average="macro", zero_division=0),
        "converged": model.gmm_.converged_, "iterations": model.gmm_.n_iter_,
    })
    probabilities = model.predict_proba(irisX[valid_index])
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
    np.testing.assert_array_equal(
        predicted, model.classes_[np.argmax(probabilities, axis=1)]
    )
    # 不同类别的映射必须是一一对应，且每个训练样本只计数一次。
    assert len(np.unique(model.component_to_class_)) == len(model.classes_)
    assert model.contingency_.sum() == len(train_index)
    trained_models.append(model)

fold_results = pd.DataFrame(records).set_index("fold")
metric_names = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
cv_mean = fold_results[metric_names].mean()
cv_std = fold_results[metric_names].std(ddof=1)
print("逐折指标与 EM 收敛情况")
display(fold_results.round(6))
print("十折均值及样本标准差")
display(pd.DataFrame({"mean": cv_mean, "std": cv_std}).round(6))
labels = np.unique(irisY)
print("折外预测混淆矩阵；行是真实类别，列是预测类别")
display(pd.DataFrame(
    confusion_matrix(irisY, oof_prediction, labels=labels), index=labels, columns=labels
))
assert len(records) == 10 and fold_results["converged"].all()
print("验证通过：所有折的 GMM 均收敛，类别映射及概率归一化检查通过。")


逐折指标与 EM 收敛情况


,accuracy,precision_macro,recall_macro,f1_macro,converged,iterations
fold,,,,,,
1,1.000000,1.000000,1.000000,1.000000,True,33
2,1.000000,1.000000,1.000000,1.000000,True,6
3,0.933333,0.944444,0.933333,0.932660,True,11
4,1.000000,1.000000,1.000000,1.000000,True,10
5,0.933333,0.944444,0.933333,0.932660,True,8
6,0.866667,0.904762,0.866667,0.861111,True,9
7,1.000000,1.000000,1.000000,1.000000,True,6
8,1.000000,1.000000,1.000000,1.000000,True,21
9,0.933333,0.944444,0.933333,0.932660,True,15


十折均值及样本标准差


,mean,std
accuracy,0.960000,0.046614
precision_macro,0.968254,0.035493
recall_macro,0.960000,0.046614
f1_macro,0.959175,0.048030


折外预测混淆矩阵；行是真实类别，列是预测类别


,Iris-setosa,Iris-versicolor,Iris-virginica
Iris-setosa,50,0,0
Iris-versicolor,0,44,6
Iris-virginica,0,0,50


验证通过：所有折的 GMM 均收敛，类别映射及概率归一化检查通过。


In [4]:
# 展示第一折的训练集分量映射；此表没有使用该折验证标签。
first_model = trained_models[0]
print("第一折训练集上的分量 × 类别计数")
display(pd.DataFrame(
    first_model.contingency_,
    index=[f"分量 {k}" for k in range(len(first_model.classes_))],
    columns=first_model.classes_,
))
display(pd.DataFrame({
    "component": np.arange(len(first_model.classes_)),
    "class": first_model.component_to_class_,
    "mixture_weight": first_model.gmm_.weights_,
}).set_index("component"))


第一折训练集上的分量 × 类别计数


,Iris-setosa,Iris-versicolor,Iris-virginica
分量 0,0,5,45
分量 1,0,40,0
分量 2,45,0,0


,class,mixture_weight
component,,
0,Iris-virginica,0.372251
1,Iris-versicolor,0.294416
2,Iris-setosa,0.333333


## 结果与分析

| 指标 | 十折均值 | 样本标准差 |
| --- | --- | --- |
| accuracy | 0.960000 | 0.046614 |
| precision_macro | 0.968254 | 0.035493 |
| recall_macro | 0.960000 | 0.046614 |
| f1_macro | 0.959175 | 0.048030 |

十折平均准确率为 0.960000，宏平均查准率为 0.968254，宏平均查全率为 0.960000，宏平均 F1 为 0.959175。全部十折均收敛，各折最优初始化的 EM 迭代次数最多为 33 次。类别映射只使用训练集标签，所报告指标来自未参与该折拟合和映射的验证样本。混淆矩阵用于查看具体类别间的错误；分量编号可随初始化改变，因此通过训练标签映射后评价分类指标才有意义。

GMM 在 EM 拟合阶段不使用类别标签，因此其目标是提高特征分布的似然，未直接优化分类准确率。
一个高斯分量也未必恰好对应一个真实类别。多次初始化用于降低局部最优的影响，
`reg_covar` 用于避免协方差奇异；这些设置均事先固定，没有使用验证集选择参数。
